<b><font size="6" color="#E8800A">Week 13a: Choosing neural-network settings</font></b><br>
<b><font size="4">How the network is fitted</font></b><br>

This class keeps the layers and activations of the Week 12 joint network and changes how it is fitted: the solver, the L2 penalty, the learning rate and its schedule, and the stopping rule. A random search then combines these settings with the Week 12 choices.

<div class="alert alert-block alert-info">

# TOC<a class="anchor" id="toc"></a>
* [<font color='#E8800A'>Setup</font>](#setup)
* [<font color='#E8800A'>Optimisation changes the fitted network</font>](#optimisation)
* [<font color='#E8800A'>How backpropagation computes the gradient</font>](#backprop)
* [<font color='#E8800A'>Assess the unchanged reference</font>](#reference)
* [<font color='#E8800A'>MLP settings</font>](#settings)
* [<font color='#E8800A'>Nested cross-validation</font>](#nested)
* [<font color='#E8800A'>The final search and its winner</font>](#search)
* [<font color='#E8800A'>The course board</font>](#board)
* [<font color='#E8800A'>Key takeaways</font>](#takeaways)
* [<font color='#E8800A'>References</font>](#references)

</div>

# <font color='#E8800A'>Setup</font> <a class="anchor" id="setup"></a>
[Back to TOC](#toc)

In [ ]:
import sys
import joblib
from pathlib import Path
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

import time
import warnings
from functools import partial
from joblib import parallel_config
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from matplotlib_inline.backend_inline import set_matplotlib_formats
from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import balanced_accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, ParameterGrid, StratifiedKFold, cross_validate, train_test_split
from sklearn.neural_network import MLPClassifier
from tqdm.auto import tqdm
from course_helpers import PLOT_BLUE, PLOT_ORANGE, Scoreboard, record, results_before, result_context


__Step 1:__ Import the course preprocessing tools.

In [ ]:
random_state = 33

from preprocessing import (
    CachedSearchCV,
    FoldPreprocessor,
    PreparedEstimator,
    load_recipe,
    outer_folds,
    outer_splitter,
    prepared_folds,
)
classification_recipe_log = "../../logs/week_04_feature_work_classification_log.json"

__Step 2:__ Set the seed, configure the plots and filter the expected warnings.

In [ ]:
np.random.seed(random_state)
sns.set_theme(style="whitegrid")
set_matplotlib_formats("svg")
plt.rcParams["figure.dpi"] = 150

ignore_unknown_category_warning = partial(
    warnings.filterwarnings, "ignore",
    message=r"^Found unknown categories in columns \[.*\] during transform\. These unknown categories will be encoded as all zeros$",
    category=UserWarning, module=r"^sklearn\.preprocessing\._encoders$")
ignore_unknown_category_warning()
def search_warning_policy():
    ignore_unknown_category_warning()
    warnings.filterwarnings("ignore",
        message=r"^Stochastic Optimizer: Maximum iterations \((20|200|1000|2000)\) reached and the optimization hasn't converged yet\.$",
        category=ConvergenceWarning, module=r"^sklearn\.neural_network\._multilayer_perceptron$")
    warnings.filterwarnings("ignore",
        message=r"^lbfgs failed to converge after (?P<mlp_cap>20|200|1000|2000) iteration\(s\) \(status=1\):\nSTOP: TOTAL NO\. OF ITERATIONS REACHED LIMIT\n\nIncrease the number of iterations to improve the convergence \(max_iter=(?P=mlp_cap)\)\.\nYou might also want to scale the data as shown in:\n    https://scikit-learn\.org/stable/modules/preprocessing\.html$",
        category=ConvergenceWarning, module=r"^sklearn\.neural_network\._multilayer_perceptron$")

__Step 3:__ Load the athletes, set aside the test rows, and prepare the training folds.

In [ ]:
with parallel_config(backend='loky', initializer=ignore_unknown_category_warning):
    champions, classification_recipe = load_recipe('../../data/interim/champions.csv', classification_recipe_log)
    inherited_pipeline = joblib.load('../../results/week_12_classification_model.joblib')
    classification_recipe = inherited_pipeline.named_steps['preparation'].recipe
    champions = champions.set_index('RecordID')
    X = champions[classification_recipe['categorical'] + classification_recipe['numeric']]
    y = champions['Outcome']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=random_state)
    fixed_settings = inherited_pipeline.named_steps['model'].get_params(deep=False)
    mlp_prep = FoldPreprocessor(classification_recipe).fit(X_train, y_train)
    Z_train = mlp_prep.transform(X_train)
    Z_test = mlp_prep.transform(X_test)
    mlp_folds = outer_splitter('classification', random_state=random_state)
    mlp_prepared = prepared_folds(FoldPreprocessor(classification_recipe), mlp_folds, X_train, y_train, n_jobs=2)
    mlp_log = []
    print(f'{len(y_train):,} training athletes; {len(y_test):,} held-out athletes')

### The Week 12 winner

This is the fitted network saved at the end of Week 12. Its weights, layer sizes and activations provide the starting reference for the optimisation comparisons.

__Step 4:__ Read the fitted arrays and the activation function of each layer, and choose the units to draw.

In [ ]:
# The fitted network's own arrays. A layer wider than six units shows its first five units, an ellipsis and its last unit.
net_model = inherited_pipeline.named_steps['model']
net_weights, net_biases = net_model.coefs_, net_model.intercepts_
net_sizes = [net_weights[0].shape[0]] + [weight.shape[1] for weight in net_weights]
net_shown = [list(range(size)) if size <= 6 else [*range(5), None, size - 1] for size in net_sizes]
# The activation of each layer is read from the fitted model: `activation` for the hidden layers and `out_activation_` for the output.
net_activations = ["none"] + [net_model.activation] * (len(net_sizes) - 2) + [net_model.out_activation_]
net_symbol = "p"
net_layers = pd.DataFrame({"units": net_sizes, "units drawn": [len([unit for unit in shown if unit is not None]) for shown in net_shown],
                           "activation": net_activations},
                          index=["input"] + [f"hidden {layer}" for layer in range(1, len(net_sizes) - 1)] + ["output"])
display(net_layers)

__Step 5:__ Tabulate the connections between the units drawn.

In [ ]:
# One row per unit drawn, and one row per connection between units drawn in neighbouring layers.
net_gap = 4
net_prefix = ["x"] + (["h"] if len(net_sizes) == 3 else [f"h{layer}_" for layer in range(1, len(net_sizes) - 1)])
net_math = ["x"] + (["h"] if len(net_sizes) == 3 else [f"h^{{({layer})}}" for layer in range(1, len(net_sizes) - 1)])
net_node_rows = []
for layer, slots in enumerate(net_shown):
    last = layer == len(net_sizes) - 1
    for slot, unit in enumerate(slots):
        net_node_rows.append({"layer": layer, "unit": -1 if unit is None else unit, "x": layer * net_gap, "y": 1.2 * ((len(slots) - 1) / 2 - slot),
                              "name": "output" if last else f"{net_prefix[layer]}{unit + 1}" if unit is not None else "",
                              "tex": f"${net_symbol}$" if last else f"${net_math[layer]}_{{{unit + 1}}}$" if unit is not None else ""})
net_nodes = pd.DataFrame(net_node_rows)
net_circles = net_nodes[net_nodes["unit"] >= 0]
net_link_rows = []
for layer, weights in enumerate(net_weights):
    for source in net_circles[net_circles["layer"] == layer].itertuples():
        for target in net_circles[net_circles["layer"] == layer + 1].itertuples():
            net_link_rows.append({"from": source.name, "to": target.name, "weight": weights[source.unit, target.unit],
                                  "x0": source.x, "y0": source.y, "x1": target.x, "y1": target.y})
net_links = pd.DataFrame(net_link_rows)
net_largest = max(np.abs(weight).max() for weight in net_weights)
# The text of the figure: a title for each layer with its activation function, and the shapes of each pair of arrays.
net_hidden_formula = {"relu": r"\max(0,z)", "tanh": r"\tanh(z)", "logistic": r"\sigma(z)", "identity": "z"}[net_model.activation]
net_output_formula = {"logistic": r"\sigma(a)", "identity": "a"}[net_model.out_activation_]
net_hidden_names = ["hidden layer" if len(net_sizes) == 3 else f"hidden layer {layer}" for layer in range(1, len(net_sizes) - 1)]

__Step 6:__ Label each layer with its fitted activation and array shapes.

In [ ]:
net_titles = ([f"inputs $x$\n{net_sizes[0]} units"]
              + [f"{name}, {size} units\n{net_model.activation}: $h={net_hidden_formula}$" for name, size in zip(net_hidden_names, net_sizes[1:-1])]
              + [f"output\n{net_model.out_activation_}: ${net_symbol}={net_output_formula}$" + ""])
net_captions = [f"$W_{{{k}}}$ ({w.shape[0]}×{w.shape[1]}), $b_{{{k}}}$ ({len(b)})" for k, (w, b) in enumerate(zip(net_weights, net_biases), 1)]
net_face = ["#EDEDED"] + ["#FFF3C4"] * (len(net_sizes) - 2) + ["#DFF5E8"]
print(f"{len(net_links)} connections drawn; largest absolute weight of the network {net_largest:.3f}")
# The ten largest of the connections drawn.
display(net_links.assign(size=net_links["weight"].abs()).nlargest(10, "size")[["from", "to", "weight"]].round(3).reset_index(drop=True))

__Step 7:__ Draw the network from that table.

In [ ]:
net_blue, net_orange = "#1779A8", "#C96A00"
net_fig, net_ax = plt.subplots(figsize=(min(14, 2.6 * len(net_sizes) + 2), 7.4))
for link in net_links.itertuples():
    net_size = np.abs(link.weight) / net_largest
    net_ax.plot([link.x0, link.x1], [link.y0, link.y1], color=net_blue if link.weight > 0 else net_orange,
                lw=0.4 + 4.6 * net_size, alpha=0.12 + 0.8 * net_size, solid_capstyle="round", zorder=1)
net_ax.scatter(net_circles["x"], net_circles["y"], s=560, facecolor=[net_face[layer] for layer in net_circles["layer"]],
               edgecolor="#222222", linewidth=1.2, zorder=3)
for node in net_nodes.itertuples():
    net_ax.text(node.x, node.y, node.tex if node.unit >= 0 else r"$\vdots$", ha="center", va="center",
                fontsize=11 if node.unit >= 0 else 18, zorder=4)
net_top, net_bottom = net_nodes["y"].max(), net_nodes["y"].min()
for layer, title in enumerate(net_titles):
    net_ax.text(layer * net_gap, net_top + 0.9, title, ha="center", va="bottom", fontweight="bold", fontsize=12)
for gap, caption in enumerate(net_captions):
    net_ax.text((gap + 0.5) * net_gap, net_bottom - 1.0, caption, ha="center", va="top", fontsize=12)
# Two keys: the sign, and three sample widths.
net_sign_key = net_ax.legend(handles=[net_ax.plot([], [], color=net_blue, lw=3, label="positive")[0], net_ax.plot([], [], color=net_orange, lw=3, label="negative")[0]],
                             title="weight sign", loc="lower left", bbox_to_anchor=(0.0, -0.06), ncol=2, frameon=False, fontsize=11, title_fontsize=11)
net_ax.add_artist(net_sign_key)
net_ax.legend(handles=[net_ax.plot([], [], color="#555555", lw=0.4 + 4.6 * size, label=f"{size * net_largest:.2f}")[0] for size in (0.25, 0.5, 1.0)],
              title="weight size", loc="lower right", bbox_to_anchor=(1.0, -0.06), ncol=3, frameon=False, fontsize=11, title_fontsize=11)
net_ax.set(xlim=(-1, net_gap * (len(net_sizes) - 1) + 1.4), ylim=(net_bottom - 2.8, net_top + 3.4), title="The Week 12 winner")
net_ax.axis("off")
plt.tight_layout()

__Step 8:__ Show the figure.

In [ ]:
plt.show()

# <font color='#E8800A'>Optimisation changes the fitted network</font> <a class="anchor" id="optimisation"></a>
[Back to TOC](#toc)

Week 12 combined the best value of each sweep into one network. That joint network and its complete preparation remain the reference here. The forward calculation stays the same; the comparisons now change how the network is fitted.

`solver='sgd'` ([Robbins and Monro, 1951](https://doi.org/10.1214/aoms/1177729586)) updates weights using gradients from batches of training rows. `adam` ([Kingma and Ba, 2015](https://arxiv.org/abs/1412.6980)) adapts the size of individual weight updates using recent gradients. Both differentiate the training loss, not F1. `lbfgs` ([Liu and Nocedal, 1989](https://doi.org/10.1007/BF01589116)) uses the whole training matrix and approximates curvature to choose its updates. A batch size and a stochastic learning-rate schedule therefore have no role for L-BFGS.

Binary prediction loss is $-\frac{1}{n}\sum_i[y_i\log p_i+(1-y_i)\log(1-p_i)]$. Here $p_i$ is the predicted probability of class 1 for row $i$ of $n$, and $y_i$ its 0 or 1 label. `log_loss` computes it on both partitions; F1 scores the thresholded predictions separately.

Short iteration allowances leave optimisation unfinished. The diagnostic table counts the iteration-limit warnings of each fit below, the full-training fits included.

In [ ]:
fit_diagnostics = []
def note_fit(estimator, caught, label):
    expected = []
    for item in caught:
        known = (issubclass(item.category, ConvergenceWarning)
                 and ("Maximum iterations" in str(item.message)
                      or "TOTAL NO. OF ITERATIONS REACHED LIMIT" in str(item.message)))
        if known:
            expected.append(item)
        else:
            warnings.warn_explicit(item.message, item.category, item.filename, item.lineno)
    network = estimator.regressor_ if hasattr(estimator, "regressor_") else estimator
    fit_diagnostics.append({"fit": label, "iterations": network.n_iter_,
                            "iteration allowance": network.max_iter,
                            "convergence warnings": len(expected),
                            "status": ("iteration limit" if expected else "other convergence warning" if any(issubclass(w.category, ConvergenceWarning) for w in caught) else "stopping rule satisfied")})

In [ ]:
from sklearn.metrics import log_loss

def prediction_diagnostics(estimator, fold):
    losses, scores = [], []
    for partition in ["fit", "score"]:
        features, target = fold[f"X_{partition}"], fold[f"y_{partition}"]
        probability = estimator.predict_proba(features)
        losses.append(log_loss(target, probability, labels=estimator.classes_))
        scores.append(f1_score(target, estimator.predict(features)))
    return [*losses, *scores]

`alpha` weights L2 regularisation, the squared-weight penalty added to the training objective:

$$L = L_{\mathrm{classification}} + \frac{\alpha}{2n}\sum_j w_j^2.$$

Here $L$ is the regularised training objective and $L_{\mathrm{classification}}$ is mean cross-entropy, the penalty for incorrect class probabilities. The symbol $n$ counts rows in this loss calculation: a minibatch for a stochastic update and the full fitting partition for L-BFGS. The index $j$ runs over all network weights $w_j$. Larger `alpha` makes large weights more costly. `learning_rate_init` sets the initial stochastic update size, while `batch_size` sets how many rows contribute to each such update. A small batch changes both the gradient noise and the number of updates in an epoch.

### Follow each solver iteration

A single input and no hidden layer leave one weight and one bias to learn. The left panel shows their position on the loss surface; the right panel shows training and validation binary cross-entropy at that position. Each frame is a deterministic fit with the iteration allowance shown, using seed 33. The toy uses 24 training points and 18 separate validation points; solid lines show training loss and dashed lines validation loss.

SGD and Adam count passes through a full batch in this example. L-BFGS counts quasi-Newton iterations and can evaluate several trial steps per iteration. Equal iteration counts therefore represent different amounts of work. After a solver stops, its dot stays at the final position and its status says stopped.

![Solver positions and prediction loss at each iteration](../../assets/week_13/solver_classification.gif)

# <font color='#E8800A'>How backpropagation computes the gradient</font> <a class="anchor" id="backprop"></a>
[Back to TOC](#toc)

Backpropagation applies the chain rule to the training loss. This example has two inputs, two ReLU hidden units and one output: four input weights, two hidden biases, two output weights and one output bias. The following cells evaluate each stage using the supplied arrays.

Rows of $W_1$ index inputs and columns index hidden units. With one input row, the hidden weighted sums are $z=xW_1+b_1$, their activations are $h=\max(0,z)$, and the output weighted sum is $a=hW_2+b_2$. The infographic states the array shapes and distinguishes matrix multiplication from the elementwise ReLU derivative.

Classification predicts $p=\sigma(a)$ and uses $L=-y\log p-(1-y)\log(1-p)$. Its output derivative is $\partial L/\partial a=p-y$. The toy target is $y=1$ and the learning rate is $\eta=0.1$; these are arbitrary toy units, without an L2 penalty. Every gradient uses the parameters before the update. Apply all updates simultaneously, then recompute the prediction and loss.

### Follow the calculation through the graph

Each stage places the formula above its numerical substitution. Forward arrows carry values; backward arrows carry derivatives. The fixed example uses the arrays in the calculations below. Its final frames apply all parameter updates simultaneously and recompute the loss.

Matrix multiplication, written $@$ in NumPy, sums over the shared dimension. Here $xW_1$ has shape $(1,2)@(2,2)\to(1,2)$. The Hadamard product, $\odot$ or NumPy $*$, multiplies corresponding entries without summing. Backpropagation first computes $\delta_o W_2^T:(1,1)@(1,2)\to(1,2)$, then applies the ReLU derivative elementwise: $\delta_h=(\delta_o W_2^T)\odot\mathbf1[z>0]$. The weight gradient $x^T\delta_h$ is an outer matrix product, $(2,1)@(1,2)\to(2,2)$. For two-dimensional arrays, `np.dot(A, B)` and `A @ B` perform matrix multiplication; neither is the Hadamard product. A vector dot product that reduced the hidden derivatives to one scalar would lose the separate gradient for each hidden unit.

The animation follows the operation-by-operation presentation in [Stanford CS224N, neural networks and backpropagation](https://web.stanford.edu/class/cs224n/slides/cs224n-spr2024-lecture03-neuralnets.pdf). Its numbers use this notebook's two-unit example. Blue plus FORWARD denotes value computation; red plus BACKWARD denotes differentiation.

![Computational graph with formulas above numerical substitutions](../../assets/week_13/backprop_classification.gif)

### Parameters and shapes

The two columns of W1 belong to hidden units 1 and 2. There are four input weights, two hidden biases, two output weights and one output bias: nine scalar parameters.

In [ ]:
bp_x = np.array([[1.0, 2.0]])
bp_W1 = np.array([[0.1, -0.2], [0.2, 0.2]])
bp_b1 = np.array([0.1, 0.0])
bp_W2 = np.array([[0.5], [0.5]])
bp_b2 = np.array([0.0])
bp_y = 1.0
bp_eta = 0.1
print("Input and parameter shapes:", bp_x.shape, bp_W1.shape, bp_b1.shape, bp_W2.shape, bp_b2.shape)

### Hidden activations

$z_1=1(0.1)+2(0.2)+0.1=0.6$ and $z_2=1(-0.2)+2(0.2)+0=0.2$. Both are positive, so $h=[0.6,0.2]$. Both sums are away from zero, the point where ReLU has no derivative.

In [ ]:
bp_z = bp_x @ bp_W1 + bp_b1
bp_h = np.maximum(0.0, bp_z)
print("Hidden sums z:", bp_z)
print("ReLU activations h:", bp_h)

### Prediction and loss

$a=0.6(0.5)+0.2(0.5)+0=0.4$. The probability is $p=\sigma(a)=1/(1+e^{-a})=0.5986876601$. Binary cross-entropy is $L=-y\log p-(1-y)\log(1-p)=0.5130152524$. This one-example loss has no averaging or regularisation term.

In [ ]:
bp_a = bp_h @ bp_W2 + bp_b2
bp_prediction = 1.0 / (1.0 + np.exp(-bp_a))
bp_loss = (-bp_y * np.log(bp_prediction) - (1.0-bp_y) * np.log1p(-bp_prediction)).item()
print("Output weighted sum:", bp_a)
print("Probability:", bp_prediction)
print("Binary cross-entropy:", bp_loss)

### Output derivative

$\partial L/\partial p=-y/p+(1-y)/(1-p)$ and $\partial p/\partial a=p(1-p)$. Their product simplifies to $\delta_{out}=\partial L/\partial a=p-y=-0.4013123399$. This signed loss signal is not the probability $p=0.5986876601$. Retain its $(1,1)$ shape for the matrix products.

In [ ]:
bp_dL_dp = -bp_y / bp_prediction + (1.0-bp_y) / (1.0-bp_prediction)
bp_dp_da = bp_prediction * (1.0-bp_prediction)
bp_delta_out = bp_dL_dp * bp_dp_da
print("dL/dp:", bp_dL_dp, "dp/da:", bp_dp_da)
print("Output loss signal:", bp_delta_out, "shape:", bp_delta_out.shape)
assert np.allclose(bp_delta_out, bp_prediction-bp_y)

### Output-weight gradients

The gradients $dW_2=\partial L/\partial W_2$ and $db_2=\partial L/\partial b_2$ are arrays with the same shape as the parameter they belong to, so $(2,1)$ and $(1,)$. The transpose $h^T$ turns the $(1,2)$ row $h$ into a $(2,1)$ column, and `.T` is the NumPy spelling, so $dW_2=h^T\delta_{out}$ has shape $(2,1)$. The bias derivative is $\delta_{out}$ summed over its single row, which is what `.sum(axis=0)` does, and it has shape $(1,)$. The two weight derivatives are $0.6(-0.4013123399)=-0.2407874039$ and $0.2(-0.4013123399)=-0.0802624680$; the bias derivative is $-0.4013123399$.

In [ ]:
bp_dW2 = bp_h.T @ bp_delta_out
bp_db2 = bp_delta_out.sum(axis=0)
print("dW2:", bp_dW2, "db2:", bp_db2)

### Hidden derivatives

$\delta_{hidden}=(\delta_{out}W_2^T)\odot\mathbf{1}(z>0)$, where $\odot$ means elementwise multiplication and $\mathbf{1}(z>0)$ is the ReLU derivative. For unit 1, $-0.4013123399(0.5)(1)=-0.2006561699$. Unit 2 receives the same signal. A negative hidden sum would give a zero ReLU derivative and block the derivative through that unit. Neither sum here is negative.

In [ ]:
bp_delta_hidden = (bp_delta_out @ bp_W2.T) * (bp_z > 0)
print("Hidden loss signals:", bp_delta_hidden)

### Input-weight gradients

Likewise $dW_1=\partial L/\partial W_1$ and $db_1=\partial L/\partial b_1$ have the shapes of $W_1$ and $b_1$. Here $x^T$ is a $(2,1)$ column, so $dW_1=x^T\delta_{hidden}$ has shape $(2,2)$, and summing the hidden signal over its single row gives $db_1$ with shape $(2,)$. The rows of $dW_1$ are $[-0.2006561699,-0.2006561699]$ and $[-0.4013123399,-0.4013123399]$ because the inputs are 1 and 2. Both bias derivatives are $-0.2006561699$.

In [ ]:
bp_dW1 = bp_x.T @ bp_delta_hidden
bp_db1 = bp_delta_hidden.sum(axis=0)
print("dW1:", bp_dW1)
print("db1:", bp_db1)

### Parameter update

Each parameter after the update equals its value before the update minus $\eta$ times its derivative, the plain gradient-descent rule behind the `sgd` solver. Compute all four gradient arrays before changing any parameter. $W_1^{new}=[[0.1200656170,-0.1799343830],[0.2401312340,0.2401312340]]$, $b_1^{new}=[0.1200656170,0.0200656170]$, $W_2^{new}=[[0.5240787404],[0.5080262468]]$, and $b_2^{new}=[0.0401312340]$. A scikit-learn `sgd` update also averages the gradient over a minibatch of rows and adds momentum by default.

In [ ]:
bp_W1_new = bp_W1-bp_eta*bp_dW1
bp_b1_new = bp_b1-bp_eta*bp_db1
bp_W2_new = bp_W2-bp_eta*bp_dW2
bp_b2_new = bp_b2-bp_eta*bp_db2
print("New W1:", bp_W1_new, "new b1:", bp_b1_new)
print("New W2:", bp_W2_new, "new b2:", bp_b2_new)

### Prediction after the update

After the update, the hidden sums are $[0.7203937020,0.3203937020]$. The output weighted sum becomes $0.5804426678$, probability becomes $0.6411692578$, and loss falls from $0.5130152524$ to $0.4444618041$.

In [ ]:
bp_z_new = bp_x @ bp_W1_new+bp_b1_new
bp_h_new = np.maximum(0.0, bp_z_new)
bp_a_new = bp_h_new @ bp_W2_new+bp_b2_new
bp_prediction_new = 1.0/(1.0+np.exp(-bp_a_new))
bp_loss_new = (-bp_y*np.log(bp_prediction_new)-(1.0-bp_y)*np.log1p(-bp_prediction_new)).item()
print("New hidden sums:", bp_z_new)
print("New output weighted sum:", bp_a_new, "new prediction:", bp_prediction_new)
print("Loss before:", bp_loss, "loss after:", bp_loss_new)
assert bp_loss_new < bp_loss

### Using the same parameter values

All derivatives must describe the same loss at the same parameter values. Updating an output weight early changes the chain-rule multiplier for the hidden layer and mixes two parameter states. Storing the updated arrays separately preserves the parameters from before the update until every derivative has been computed.

# <font color='#E8800A'>Assess the unchanged reference</font> <a class="anchor" id="reference"></a>
[Back to TOC](#toc)

Ten repeated course folds score the Week 12 joint network before any setting changes.

__Step 9:__ Score the prepared reference on the ten prepared folds.

In [ ]:
reference_train, reference_validation = ([], [])
reference_auc, reference_balanced = ([], [])
for fold in tqdm(mlp_prepared, desc='Course-fold assessment', unit='fold', leave=True, position=0):
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always', ConvergenceWarning)
        fitted_fold = clone(MLPClassifier(**fixed_settings)).fit(fold['X_fit'], fold['y_fit'])
    note_fit(fitted_fold, caught, 'reference, fold')
    reference_train.append(f1_score(fold['y_fit'], fitted_fold.predict(fold['X_fit'])))
    reference_validation.append(f1_score(fold['y_score'], fitted_fold.predict(fold['X_score'])))
    reference_auc.append(roc_auc_score(fold['y_score'], fitted_fold.predict_proba(fold['X_score'])[:, 1]))
    reference_balanced.append(balanced_accuracy_score(fold['y_score'], fitted_fold.predict(fold['X_score'])))
print('Reference validation F1 by fold:', np.round(reference_validation, 4))

__Step 10:__ Fit the reference on all training athletes.

In [ ]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always', ConvergenceWarning)
    fitted_reference = clone(MLPClassifier(**fixed_settings)).fit(Z_train, y_train)
note_fit(fitted_reference, caught, 'reference, all training rows')

__Step 11:__ Log the unchanged reference with its training, validation and held-out F1.

In [ ]:
reference = {"settings": fixed_settings, "configuration": " ".join(repr(MLPClassifier(**fixed_settings)).split()),
             "train": reference_train, "validation": reference_validation, "model": fitted_reference}
record(mlp_log, {"frame": type(mlp_folds).__name__, "rows": len(y_train),
    "folds": mlp_folds.get_n_splits(), "name": reference["configuration"],
    "train": reference["train"], "validation": reference["validation"], "model": fitted_reference},
    Z_test, y_test, "selected in Week 12", metric=f1_score, label="F1")

**The Week 12 joint reference reaches validation F1 0.8523 and test F1 0.8509, with mean training F1 0.8599.** It has one tanh layer of eight units, batch size 512 and a 200-iteration allowance.

Each checkpoint refits the network on the first prepared training fold with the iteration allowance on the horizontal axis, then scores that fold's training and validation rows. The losses omit the L2 penalty, which the optimiser also minimises.

__Step 12:__ Set the checkpoint budgets and candidate settings.

In [ ]:
curve_fold = mlp_prepared[0]
curve_budgets = [1, 3, 10, 30, 100, 300, 1000, 2000]
curve_points = {}
curve_values = ['reference']
curve_base = clone(MLPClassifier(**fixed_settings))

__Step 13:__ Fit each checkpoint once and measure both losses and task scores.

In [ ]:
for curve_value in tqdm(curve_values, desc='checkpoint values', leave=True, position=0):
    curve_rows = []
    for curve_k in curve_budgets:
        curve_settings = {"max_iter": curve_k}
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", ConvergenceWarning)
            curve_fit = clone(curve_base).set_params(**curve_settings).fit(
                curve_fold["X_fit"], curve_fold["y_fit"])
        note_fit(curve_fit, caught, f"Reference network: {curve_value}, budget {curve_k}")
        curve_row = prediction_diagnostics(curve_fit, curve_fold)
        curve_rows.append([curve_k, *curve_row])
        curve_network = curve_fit
        if curve_network.n_iter_ < curve_k:
            break
    curve_points[str(curve_value)] = np.array(curve_rows)

__Step 14:__ Compare prediction loss and task score in separate panels.

In [ ]:
curve_fig, curve_axes = plt.subplots(1, 2, figsize=(12, 4.5))
for curve_label, curve_rows in curve_points.items():
    curve_line, = curve_axes[0].plot(curve_rows[:, 0], curve_rows[:, 1], label=f"{curve_label} train")
    curve_colour = curve_line.get_color()
    curve_axes[0].plot(curve_rows[:, 0], curve_rows[:, 2], "--", color=curve_colour, label=f"{curve_label} validation")
    curve_axes[1].plot(curve_rows[:, 0], curve_rows[:, 3], color=curve_colour, label=f"{curve_label} train")
    curve_axes[1].plot(curve_rows[:, 0], curve_rows[:, 4], "--", color=curve_colour, label=f"{curve_label} validation")
for curve_ax in curve_axes:
    curve_ax.set_xscale("log")
    curve_ax.set_xlabel("iteration allowance (log scale)")
    curve_ax.legend(fontsize=8)
curve_axes[0].set(ylabel="binary cross-entropy", title="Prediction loss: training and validation")
curve_axes[1].set(ylabel="F1", title="Task score: training and validation")
curve_fig.suptitle('Reference network')
plt.tight_layout()
plt.show()
display(pd.DataFrame(fit_diagnostics).tail(sum(len(rows) for rows in curve_points.values())))

# <font color='#E8800A'>MLP settings</font> <a class="anchor" id="settings"></a>
[Back to TOC](#toc)

Each sweep changes one argument of the Week 12 reference on the same training folds, and the row with the highest mean validation F1 is logged. Every tested value enters the joint search.

| Parameter | Meaning |
| --- | --- |
| `solver` | Optimisation algorithm: Adam, SGD or L-BFGS. |
| `alpha` | Strength of the L2 penalty on the weights. |
| `learning_rate_init` | Initial step size of the stochastic solvers. |
| `n_iter_no_change` | Epochs without improvement before the fit stops. |
| `learning_rate` | Schedule of the step size under SGD. |
| `early_stopping` | Whether fitting stops when the score on an internal validation subset fails to improve. |
| `validation_fraction` | Share of the fitting rows held back for that internal check. |

[Parameter reference](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html)

### `MLP: solver`

The solvers update the same network through different optimisation rules.

`lbfgs` uses a quasi-Newton approximation to curvature on the complete fitting partition. `sgd` updates weights from minibatch gradients; `adam` adapts each weight's update using running gradient moments. The candidate list is `["adam", "sgd", "lbfgs"]`.

__Step 15:__ Set the `solver` values.

In [ ]:
# The values of this one setting; every other setting stays at the reference.
mlp_solver_values = ["adam", "sgd", "lbfgs"]
# One row per value will hold its scores, its seconds and its fitted model.
mlp_solver_rows = []

__Step 16:__ Score each candidate and keep its fitted model.

In [ ]:
for value in tqdm(mlp_solver_values, desc='solver values', leave=True, position=0):
    started = time.perf_counter()
    settings = {**fixed_settings, 'solver': value}
    train_scores, validation_scores, fold_warnings = ([], [], [])
    for fold in mlp_prepared:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter('always', ConvergenceWarning)
            fitted = MLPClassifier(**settings).fit(fold['X_fit'], fold['y_fit'])
        note_fit(fitted, caught, f'solver={value}, fold')
        train_scores.append(f1_score(fold['y_fit'], fitted.predict(fold['X_fit'])))
        validation_scores.append(f1_score(fold['y_score'], fitted.predict(fold['X_score'])))
        fold_warnings.append(any((issubclass(w.category, ConvergenceWarning) for w in caught)))
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always', ConvergenceWarning)
        deployed = MLPClassifier(**settings).fit(Z_train, y_train)
    note_fit(deployed, caught, f'solver={value}, all training rows')
    shown = {key: item for key, item in MLPClassifier(**settings).get_params().items() if item != MLPClassifier().get_params()[key] or key in ('activation', 'hidden_layer_sizes', 'max_iter')}
    mlp_solver_rows.append({'configuration': 'MLPClassifier(' + ', '.join((f'{key}={item!r}' for key, item in sorted(shown.items()))) + ')', 'setting': str(value), 'seconds': time.perf_counter() - started, 'train_F1': np.mean(train_scores), 'cv_F1': np.mean(validation_scores), 'test_F1': f1_score(y_test, deployed.predict(Z_test)), 'iterations': deployed.n_iter_, 'loss': deployed.loss_, 'fold warnings': sum(fold_warnings), 'train': train_scores, 'validation': validation_scores, 'parameters': settings, 'model': deployed})

__Step 17:__ Assemble and inspect the candidate scores.

In [ ]:
# The sweep table: one row per value, with the mean fold scores, the test score, the fit diagnostics and the seconds spent.
mlp_solver_sweep = pd.DataFrame(mlp_solver_rows)
display(mlp_solver_sweep.set_index("setting")[["train_F1", "cv_F1", "test_F1", "iterations", "loss", "fold warnings", "seconds"]].round(4))
display(pd.DataFrame(fit_diagnostics).tail(11))

In [ ]:
# The sweep's winner joins the log.
mlp_solver_winner = mlp_solver_sweep.loc[mlp_solver_sweep["cv_F1"].idxmax()]
record(mlp_log, {"frame": type(mlp_folds).__name__, "rows": len(y_train), "folds": mlp_folds.get_n_splits(),
    "name": mlp_solver_winner["configuration"], "train": mlp_solver_winner["train"],
    "validation": mlp_solver_winner["validation"], "model": mlp_solver_winner["model"]},
    Z_test, y_test, "one setting changed", metric=f1_score, label="F1")

__Step 18:__ Plot the training and validation F1 for `solver`.

In [ ]:
# Training and validation F1 for each solver value.
parameter_positions = np.arange(len(mlp_solver_sweep))
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(parameter_positions, mlp_solver_sweep["train_F1"], marker="o", color=PLOT_ORANGE, label="training")
ax.plot(parameter_positions, mlp_solver_sweep["cv_F1"], marker="o", color=PLOT_BLUE, label="validation")
ax.set(title="solver", xticks=parameter_positions, xticklabels=mlp_solver_sweep["setting"], ylabel="F1")
ax.tick_params(axis="x", labelrotation=20)
ax.legend()
plt.tight_layout()
plt.show()

**Adam has validation F1 0.8523, SGD 0.8096 and L-BFGS 0.8230 at the 200-iteration allowance.** L-BFGS fits the training folds most closely, at F1 0.9167, but has test F1 0.8290 against 0.8509 for Adam. All three solvers raise convergence warnings in all ten folds.

__Step 19:__ Set the checkpoint budgets and candidate settings.

In [ ]:
curve_fold = mlp_prepared[0]
curve_budgets = [1, 3, 10, 30, 100, 300, 1000, 2000]
curve_points = {}
curve_values = mlp_solver_values
curve_base = clone(MLPClassifier(**fixed_settings))

__Step 20:__ Fit each checkpoint once and measure both losses and task scores.

In [ ]:
for curve_value in tqdm(curve_values, desc='checkpoint values', leave=True, position=0):
    curve_rows = []
    for curve_k in curve_budgets:
        curve_settings = {"solver": curve_value, "max_iter": curve_k}
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", ConvergenceWarning)
            curve_fit = clone(curve_base).set_params(**curve_settings).fit(
                curve_fold["X_fit"], curve_fold["y_fit"])
        note_fit(curve_fit, caught, f"solver: {curve_value}, budget {curve_k}")
        curve_row = prediction_diagnostics(curve_fit, curve_fold)
        curve_rows.append([curve_k, *curve_row])
        curve_network = curve_fit
        if curve_network.n_iter_ < curve_k:
            break
    curve_points[str(curve_value)] = np.array(curve_rows)

__Step 21:__ Compare prediction loss and task score in separate panels.

In [ ]:
curve_fig, curve_axes = plt.subplots(1, 2, figsize=(12, 4.5))
for curve_label, curve_rows in curve_points.items():
    curve_line, = curve_axes[0].plot(curve_rows[:, 0], curve_rows[:, 1], label=f"{curve_label} train")
    curve_colour = curve_line.get_color()
    curve_axes[0].plot(curve_rows[:, 0], curve_rows[:, 2], "--", color=curve_colour, label=f"{curve_label} validation")
    curve_axes[1].plot(curve_rows[:, 0], curve_rows[:, 3], color=curve_colour, label=f"{curve_label} train")
    curve_axes[1].plot(curve_rows[:, 0], curve_rows[:, 4], "--", color=curve_colour, label=f"{curve_label} validation")
for curve_ax in curve_axes:
    curve_ax.set_xscale("log")
    curve_ax.set_xlabel("iteration allowance (log scale)")
    curve_ax.legend(fontsize=8)
curve_axes[0].set(ylabel="binary cross-entropy", title="Prediction loss: training and validation")
curve_axes[1].set(ylabel="F1", title="Task score: training and validation")
curve_fig.suptitle('solver')
plt.tight_layout()
plt.show()
display(pd.DataFrame(fit_diagnostics).tail(sum(len(rows) for rows in curve_points.values())))

### `MLP: alpha`

L2 penalises weight magnitude.

`alpha=0` removes the L2 penalty. Larger positive values penalise large weights more strongly, which can reduce overfitting but can also underfit; the data-loss term and sample count affect the balance. The candidate list is `[1e-4, 1e-2, 1.0]`.

__Step 22:__ Set the `alpha` values.

In [ ]:
# The values of this one setting; every other setting stays at the reference.
mlp_alpha_values = [1e-4, 1e-2, 1.0]
# One row per value will hold its scores, its seconds and its fitted model.
mlp_alpha_rows = []

__Step 23:__ Score each candidate and keep its fitted model.

In [ ]:
for value in tqdm(mlp_alpha_values, desc='alpha values', leave=True, position=0):
    started = time.perf_counter()
    settings = {**fixed_settings, 'alpha': value}
    train_scores, validation_scores, fold_warnings = ([], [], [])
    for fold in mlp_prepared:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter('always', ConvergenceWarning)
            fitted = MLPClassifier(**settings).fit(fold['X_fit'], fold['y_fit'])
        note_fit(fitted, caught, f'alpha={value}, fold')
        train_scores.append(f1_score(fold['y_fit'], fitted.predict(fold['X_fit'])))
        validation_scores.append(f1_score(fold['y_score'], fitted.predict(fold['X_score'])))
        fold_warnings.append(any((issubclass(w.category, ConvergenceWarning) for w in caught)))
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always', ConvergenceWarning)
        deployed = MLPClassifier(**settings).fit(Z_train, y_train)
    note_fit(deployed, caught, f'alpha={value}, all training rows')
    shown = {key: item for key, item in MLPClassifier(**settings).get_params().items() if item != MLPClassifier().get_params()[key] or key in ('activation', 'hidden_layer_sizes', 'max_iter')}
    mlp_alpha_rows.append({'configuration': 'MLPClassifier(' + ', '.join((f'{key}={item!r}' for key, item in sorted(shown.items()))) + ')', 'setting': str(value), 'seconds': time.perf_counter() - started, 'train_F1': np.mean(train_scores), 'cv_F1': np.mean(validation_scores), 'test_F1': f1_score(y_test, deployed.predict(Z_test)), 'iterations': deployed.n_iter_, 'loss': deployed.loss_, 'fold warnings': sum(fold_warnings), 'train': train_scores, 'validation': validation_scores, 'parameters': settings, 'model': deployed})

__Step 24:__ Assemble and inspect the candidate scores.

In [ ]:
# The sweep table: one row per value, with the mean fold scores, the test score, the fit diagnostics and the seconds spent.
mlp_alpha_sweep = pd.DataFrame(mlp_alpha_rows)
display(mlp_alpha_sweep.set_index("setting")[["train_F1", "cv_F1", "test_F1", "iterations", "loss", "fold warnings", "seconds"]].round(4))
display(pd.DataFrame(fit_diagnostics).tail(11))

In [ ]:
# The sweep's winner joins the log.
mlp_alpha_winner = mlp_alpha_sweep.loc[mlp_alpha_sweep["cv_F1"].idxmax()]
record(mlp_log, {"frame": type(mlp_folds).__name__, "rows": len(y_train), "folds": mlp_folds.get_n_splits(),
    "name": mlp_alpha_winner["configuration"], "train": mlp_alpha_winner["train"],
    "validation": mlp_alpha_winner["validation"], "model": mlp_alpha_winner["model"]},
    Z_test, y_test, "one setting changed", metric=f1_score, label="F1")

__Step 25:__ Plot the training and validation F1 for `alpha`.

In [ ]:
# Training and validation F1 for each alpha value.
parameter_positions = np.arange(len(mlp_alpha_sweep))
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(parameter_positions, mlp_alpha_sweep["train_F1"], marker="o", color=PLOT_ORANGE, label="training")
ax.plot(parameter_positions, mlp_alpha_sweep["cv_F1"], marker="o", color=PLOT_BLUE, label="validation")
ax.set(title="alpha", xticks=parameter_positions, xticklabels=mlp_alpha_sweep["setting"], ylabel="F1")
ax.tick_params(axis="x", labelrotation=20)
ax.legend()
plt.tight_layout()
plt.show()

**Validation F1 is 0.8523 at both `alpha=0.0001` and `alpha=0.01`, to four decimals, and 0.8501 at 1.0.** The largest penalty lowers training F1 from 0.8599 to 0.8580 and test F1 from 0.8509 to 0.8440. All full fits use 200 iterations.

### `MLP: learning_rate_init`

`learning_rate_init` sets the starting step size of Adam and SGD; this sweep uses Adam.

A larger rate takes larger steps, which may reduce loss faster or overshoot a minimum. The candidate list is `[1e-3, 3e-3, 1e-2]`.

__Step 26:__ Set the `learning_rate_init` values.

In [ ]:
# The values of this one setting; every other setting stays at the reference.
mlp_learning_rate_init_values = [1e-3, 3e-3, 1e-2]
# One row per value will hold its scores, its seconds and its fitted model.
mlp_learning_rate_init_rows = []

__Step 27:__ Score each candidate and keep its fitted model.

In [ ]:
for value in tqdm(mlp_learning_rate_init_values, desc='learning_rate_init values', leave=True, position=0):
    started = time.perf_counter()
    settings = {**fixed_settings, 'learning_rate_init': value}
    train_scores, validation_scores, fold_warnings = ([], [], [])
    for fold in mlp_prepared:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter('always', ConvergenceWarning)
            fitted = MLPClassifier(**settings).fit(fold['X_fit'], fold['y_fit'])
        note_fit(fitted, caught, f'learning_rate_init={value}, fold')
        train_scores.append(f1_score(fold['y_fit'], fitted.predict(fold['X_fit'])))
        validation_scores.append(f1_score(fold['y_score'], fitted.predict(fold['X_score'])))
        fold_warnings.append(any((issubclass(w.category, ConvergenceWarning) for w in caught)))
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always', ConvergenceWarning)
        deployed = MLPClassifier(**settings).fit(Z_train, y_train)
    note_fit(deployed, caught, f'learning_rate_init={value}, all training rows')
    shown = {key: item for key, item in MLPClassifier(**settings).get_params().items() if item != MLPClassifier().get_params()[key] or key in ('activation', 'hidden_layer_sizes', 'max_iter')}
    mlp_learning_rate_init_rows.append({'configuration': 'MLPClassifier(' + ', '.join((f'{key}={item!r}' for key, item in sorted(shown.items()))) + ')', 'setting': str(value), 'seconds': time.perf_counter() - started, 'train_F1': np.mean(train_scores), 'cv_F1': np.mean(validation_scores), 'test_F1': f1_score(y_test, deployed.predict(Z_test)), 'iterations': deployed.n_iter_, 'loss': deployed.loss_, 'fold warnings': sum(fold_warnings), 'train': train_scores, 'validation': validation_scores, 'parameters': settings, 'model': deployed})

__Step 28:__ Assemble and inspect the candidate scores.

In [ ]:
# The sweep table: one row per value, with the mean fold scores, the test score, the fit diagnostics and the seconds spent.
mlp_learning_rate_init_sweep = pd.DataFrame(mlp_learning_rate_init_rows)
display(mlp_learning_rate_init_sweep.set_index("setting")[["train_F1", "cv_F1", "test_F1", "iterations", "loss", "fold warnings", "seconds"]].round(4))
display(pd.DataFrame(fit_diagnostics).tail(11))

In [ ]:
# The sweep's winner joins the log.
mlp_learning_rate_init_winner = mlp_learning_rate_init_sweep.loc[mlp_learning_rate_init_sweep["cv_F1"].idxmax()]
record(mlp_log, {"frame": type(mlp_folds).__name__, "rows": len(y_train), "folds": mlp_folds.get_n_splits(),
    "name": mlp_learning_rate_init_winner["configuration"], "train": mlp_learning_rate_init_winner["train"],
    "validation": mlp_learning_rate_init_winner["validation"], "model": mlp_learning_rate_init_winner["model"]},
    Z_test, y_test, "one setting changed", metric=f1_score, label="F1")

__Step 29:__ Plot the training and validation F1 for `learning_rate_init`.

In [ ]:
# Training and validation F1 for each learning_rate_init value.
parameter_positions = np.arange(len(mlp_learning_rate_init_sweep))
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(parameter_positions, mlp_learning_rate_init_sweep["train_F1"], marker="o", color=PLOT_ORANGE, label="training")
ax.plot(parameter_positions, mlp_learning_rate_init_sweep["cv_F1"], marker="o", color=PLOT_BLUE, label="validation")
ax.set(title="learning_rate_init", xticks=parameter_positions, xticklabels=mlp_learning_rate_init_sweep["setting"], ylabel="F1")
ax.tick_params(axis="x", labelrotation=20)
ax.legend()
plt.tight_layout()
plt.show()

**An initial rate of 0.003 gives the highest validation F1, 0.8541, against 0.8523 at 0.001 and 0.8447 at 0.01.** The largest rate raises training F1 to 0.8955 while test F1 falls to 0.8397. All three full fits use the 200 allowed iterations.

__Step 30:__ Set the checkpoint budgets and candidate settings.

In [ ]:
curve_fold = mlp_prepared[0]
curve_budgets = [1, 3, 10, 30, 100, 300, 1000, 2000]
curve_points = {}
curve_values = mlp_learning_rate_init_values
curve_base = clone(MLPClassifier(**fixed_settings))

__Step 31:__ Fit each checkpoint once and measure both losses and task scores.

In [ ]:
for curve_value in tqdm(curve_values, desc='checkpoint values', leave=True, position=0):
    curve_rows = []
    for curve_k in curve_budgets:
        curve_settings = {"learning_rate_init": curve_value, "max_iter": curve_k}
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", ConvergenceWarning)
            curve_fit = clone(curve_base).set_params(**curve_settings).fit(
                curve_fold["X_fit"], curve_fold["y_fit"])
        note_fit(curve_fit, caught, f"learning_rate_init: {curve_value}, budget {curve_k}")
        curve_row = prediction_diagnostics(curve_fit, curve_fold)
        curve_rows.append([curve_k, *curve_row])
        curve_network = curve_fit
        if curve_network.n_iter_ < curve_k:
            break
    curve_points[str(curve_value)] = np.array(curve_rows)

__Step 32:__ Compare prediction loss and task score in separate panels.

In [ ]:
curve_fig, curve_axes = plt.subplots(1, 2, figsize=(12, 4.5))
for curve_label, curve_rows in curve_points.items():
    curve_line, = curve_axes[0].plot(curve_rows[:, 0], curve_rows[:, 1], label=f"{curve_label} train")
    curve_colour = curve_line.get_color()
    curve_axes[0].plot(curve_rows[:, 0], curve_rows[:, 2], "--", color=curve_colour, label=f"{curve_label} validation")
    curve_axes[1].plot(curve_rows[:, 0], curve_rows[:, 3], color=curve_colour, label=f"{curve_label} train")
    curve_axes[1].plot(curve_rows[:, 0], curve_rows[:, 4], "--", color=curve_colour, label=f"{curve_label} validation")
for curve_ax in curve_axes:
    curve_ax.set_xscale("log")
    curve_ax.set_xlabel("iteration allowance (log scale)")
    curve_ax.legend(fontsize=8)
curve_axes[0].set(ylabel="binary cross-entropy", title="Prediction loss: training and validation")
curve_axes[1].set(ylabel="F1", title="Task score: training and validation")
curve_fig.suptitle('learning_rate_init')
plt.tight_layout()
plt.show()
display(pd.DataFrame(fit_diagnostics).tail(sum(len(rows) for rows in curve_points.values())))

### `MLP: n_iter_no_change`

Without early stopping, patience counts epochs whose training-loss improvement falls below the fixed tolerance. Increasing patience allows more such epochs before stopping. The candidate list is `[5, 10, 20]`.

__Step 33:__ Set the `n_iter_no_change` values.

In [ ]:
# The values of this one setting; every other setting stays at the reference.
mlp_n_iter_no_change_values = [5, 10, 20]
# One row per value will hold its scores, its seconds and its fitted model.
mlp_n_iter_no_change_rows = []

__Step 34:__ Score each candidate and keep its fitted model.

In [ ]:
for value in tqdm(mlp_n_iter_no_change_values, desc='n_iter_no_change values', leave=True, position=0):
    started = time.perf_counter()
    settings = {**fixed_settings, 'n_iter_no_change': value}
    train_scores, validation_scores, fold_warnings = ([], [], [])
    for fold in mlp_prepared:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter('always', ConvergenceWarning)
            fitted = MLPClassifier(**settings).fit(fold['X_fit'], fold['y_fit'])
        note_fit(fitted, caught, f'n_iter_no_change={value}, fold')
        train_scores.append(f1_score(fold['y_fit'], fitted.predict(fold['X_fit'])))
        validation_scores.append(f1_score(fold['y_score'], fitted.predict(fold['X_score'])))
        fold_warnings.append(any((issubclass(w.category, ConvergenceWarning) for w in caught)))
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always', ConvergenceWarning)
        deployed = MLPClassifier(**settings).fit(Z_train, y_train)
    note_fit(deployed, caught, f'n_iter_no_change={value}, all training rows')
    shown = {key: item for key, item in MLPClassifier(**settings).get_params().items() if item != MLPClassifier().get_params()[key] or key in ('activation', 'hidden_layer_sizes', 'max_iter')}
    mlp_n_iter_no_change_rows.append({'configuration': 'MLPClassifier(' + ', '.join((f'{key}={item!r}' for key, item in sorted(shown.items()))) + ')', 'setting': str(value), 'seconds': time.perf_counter() - started, 'train_F1': np.mean(train_scores), 'cv_F1': np.mean(validation_scores), 'test_F1': f1_score(y_test, deployed.predict(Z_test)), 'iterations': deployed.n_iter_, 'loss': deployed.loss_, 'fold warnings': sum(fold_warnings), 'train': train_scores, 'validation': validation_scores, 'parameters': settings, 'model': deployed})

__Step 35:__ Assemble and inspect the candidate scores.

In [ ]:
# The sweep table: one row per value, with the mean fold scores, the test score, the fit diagnostics and the seconds spent.
mlp_n_iter_no_change_sweep = pd.DataFrame(mlp_n_iter_no_change_rows)
display(mlp_n_iter_no_change_sweep.set_index("setting")[["train_F1", "cv_F1", "test_F1", "iterations", "loss", "fold warnings", "seconds"]].round(4))
display(pd.DataFrame(fit_diagnostics).tail(11))

In [ ]:
# The sweep's winner joins the log.
mlp_n_iter_no_change_winner = mlp_n_iter_no_change_sweep.loc[mlp_n_iter_no_change_sweep["cv_F1"].idxmax()]
record(mlp_log, {"frame": type(mlp_folds).__name__, "rows": len(y_train), "folds": mlp_folds.get_n_splits(),
    "name": mlp_n_iter_no_change_winner["configuration"], "train": mlp_n_iter_no_change_winner["train"],
    "validation": mlp_n_iter_no_change_winner["validation"], "model": mlp_n_iter_no_change_winner["model"]},
    Z_test, y_test, "one setting changed", metric=f1_score, label="F1")

__Step 36:__ Plot the training and validation F1 for `n_iter_no_change`.

In [ ]:
# Training and validation F1 for each n_iter_no_change value.
parameter_positions = np.arange(len(mlp_n_iter_no_change_sweep))
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(parameter_positions, mlp_n_iter_no_change_sweep["train_F1"], marker="o", color=PLOT_ORANGE, label="training")
ax.plot(parameter_positions, mlp_n_iter_no_change_sweep["cv_F1"], marker="o", color=PLOT_BLUE, label="validation")
ax.set(title="n_iter_no_change", xticks=parameter_positions, xticklabels=mlp_n_iter_no_change_sweep["setting"], ylabel="F1")
ax.tick_params(axis="x", labelrotation=20)
ax.legend()
plt.tight_layout()
plt.show()

**Patience 5 gives validation F1 0.8524, while 10 and 20 both give 0.8523.** Every full fit reaches 200 iterations with loss 0.3915. One fold stops before the allowance with patience 5; the other two settings raise warnings in all ten folds.

### `MLP: learning_rate`

For SGD, `constant` keeps the step fixed, `adaptive` divides it by five when progress stalls, and `invscaling` uses $\eta_t=\eta_0/(t+1)^{0.5}$ with default `power_t=0.5`. Here $t$ counts training samples seen, $\eta_0$ is `learning_rate_init`, and $\eta_t$ is the update step size after $t$ samples. A shrinking step can make improvement small while predictions remain poor. Every schedule row uses `solver='sgd'`, the only solver that reads this setting. The candidate list is `["constant", "adaptive", "invscaling"]`.

__Step 37:__ Set the `learning_rate` values.

In [ ]:
# The values of this one setting; every other setting stays at the reference.
mlp_learning_rate_values = ["constant", "adaptive", "invscaling"]
# One row per value will hold its scores, its seconds and its fitted model.
mlp_learning_rate_rows = []

__Step 38:__ Score each candidate and keep its fitted model.

In [ ]:
for value in tqdm(mlp_learning_rate_values, desc='learning_rate values', leave=True, position=0):
    started = time.perf_counter()
    settings = {**fixed_settings, 'solver': 'sgd', 'learning_rate': value}
    train_scores, validation_scores, fold_warnings = ([], [], [])
    for fold in mlp_prepared:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter('always', ConvergenceWarning)
            fitted = MLPClassifier(**settings).fit(fold['X_fit'], fold['y_fit'])
        note_fit(fitted, caught, f'learning_rate={value}, fold')
        train_scores.append(f1_score(fold['y_fit'], fitted.predict(fold['X_fit'])))
        validation_scores.append(f1_score(fold['y_score'], fitted.predict(fold['X_score'])))
        fold_warnings.append(any((issubclass(w.category, ConvergenceWarning) for w in caught)))
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always', ConvergenceWarning)
        deployed = MLPClassifier(**settings).fit(Z_train, y_train)
    note_fit(deployed, caught, f'learning_rate={value}, all training rows')
    shown = {key: item for key, item in MLPClassifier(**settings).get_params().items() if item != MLPClassifier().get_params()[key] or key in ('activation', 'hidden_layer_sizes', 'max_iter')}
    mlp_learning_rate_rows.append({'configuration': 'MLPClassifier(' + ', '.join((f'{key}={item!r}' for key, item in sorted(shown.items()))) + ')', 'setting': str(value), 'seconds': time.perf_counter() - started, 'train_F1': np.mean(train_scores), 'cv_F1': np.mean(validation_scores), 'test_F1': f1_score(y_test, deployed.predict(Z_test)), 'iterations': deployed.n_iter_, 'loss': deployed.loss_, 'fold warnings': sum(fold_warnings), 'train': train_scores, 'validation': validation_scores, 'parameters': settings, 'model': deployed})

__Step 39:__ Assemble and inspect the candidate scores.

In [ ]:
# The sweep table: one row per value, with the mean fold scores, the test score, the fit diagnostics and the seconds spent.
mlp_learning_rate_sweep = pd.DataFrame(mlp_learning_rate_rows)
display(mlp_learning_rate_sweep.set_index("setting")[["train_F1", "cv_F1", "test_F1", "iterations", "loss", "fold warnings", "seconds"]].round(4))
display(pd.DataFrame(fit_diagnostics).tail(11))

In [ ]:
# The sweep's winner joins the log.
mlp_learning_rate_winner = mlp_learning_rate_sweep.loc[mlp_learning_rate_sweep["cv_F1"].idxmax()]
record(mlp_log, {"frame": type(mlp_folds).__name__, "rows": len(y_train), "folds": mlp_folds.get_n_splits(),
    "name": mlp_learning_rate_winner["configuration"], "train": mlp_learning_rate_winner["train"],
    "validation": mlp_learning_rate_winner["validation"], "model": mlp_learning_rate_winner["model"]},
    Z_test, y_test, "one setting changed", metric=f1_score, label="F1")

__Step 40:__ Plot the training and validation F1 for `learning_rate`.

In [ ]:
# Training and validation F1 for each learning_rate value.
parameter_positions = np.arange(len(mlp_learning_rate_sweep))
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(parameter_positions, mlp_learning_rate_sweep["train_F1"], marker="o", color=PLOT_ORANGE, label="training")
ax.plot(parameter_positions, mlp_learning_rate_sweep["cv_F1"], marker="o", color=PLOT_BLUE, label="validation")
ax.set(title="learning_rate", xticks=parameter_positions, xticklabels=mlp_learning_rate_sweep["setting"], ylabel="F1")
ax.tick_params(axis="x", labelrotation=20)
ax.legend()
plt.tight_layout()
plt.show()

**Constant and adaptive SGD both give validation F1 0.8096; inverse scaling gives 0.6211.** The first two full fits reach 200 iterations and raise warnings on all ten folds. Inverse scaling stops after 20 iterations without a fold warning, with training F1 0.6182, so stopping can occur while both training and validation predictions remain poor.

__Step 41:__ Set the checkpoint budgets and candidate settings.

In [ ]:
curve_fold = mlp_prepared[0]
curve_budgets = [1, 3, 10, 30, 100, 300, 1000, 2000]
curve_points = {}
curve_values = mlp_learning_rate_values
curve_base = clone(MLPClassifier(**fixed_settings))

__Step 42:__ Fit each checkpoint once and measure both losses and task scores.

In [ ]:
for curve_value in tqdm(curve_values, desc='checkpoint values', leave=True, position=0):
    curve_rows = []
    for curve_k in curve_budgets:
        curve_settings = {"learning_rate": curve_value, "solver": "sgd", "max_iter": curve_k}
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", ConvergenceWarning)
            curve_fit = clone(curve_base).set_params(**curve_settings).fit(
                curve_fold["X_fit"], curve_fold["y_fit"])
        note_fit(curve_fit, caught, f"learning_rate: {curve_value}, budget {curve_k}")
        curve_row = prediction_diagnostics(curve_fit, curve_fold)
        curve_rows.append([curve_k, *curve_row])
        curve_network = curve_fit
        if curve_network.n_iter_ < curve_k:
            break
    curve_points[str(curve_value)] = np.array(curve_rows)

__Step 43:__ Compare prediction loss and task score in separate panels.

In [ ]:
curve_fig, curve_axes = plt.subplots(1, 2, figsize=(12, 4.5))
for curve_label, curve_rows in curve_points.items():
    curve_line, = curve_axes[0].plot(curve_rows[:, 0], curve_rows[:, 1], label=f"{curve_label} train")
    curve_colour = curve_line.get_color()
    curve_axes[0].plot(curve_rows[:, 0], curve_rows[:, 2], "--", color=curve_colour, label=f"{curve_label} validation")
    curve_axes[1].plot(curve_rows[:, 0], curve_rows[:, 3], color=curve_colour, label=f"{curve_label} train")
    curve_axes[1].plot(curve_rows[:, 0], curve_rows[:, 4], "--", color=curve_colour, label=f"{curve_label} validation")
for curve_ax in curve_axes:
    curve_ax.set_xscale("log")
    curve_ax.set_xlabel("iteration allowance (log scale)")
    curve_ax.legend(fontsize=8)
curve_axes[0].set(ylabel="binary cross-entropy", title="Prediction loss: training and validation")
curve_axes[1].set(ylabel="F1", title="Task score: training and validation")
curve_fig.suptitle('learning_rate')
plt.tight_layout()
plt.show()
display(pd.DataFrame(fit_diagnostics).tail(sum(len(rows) for rows in curve_points.values())))

### `MLP: early_stopping`

`early_stopping=True` reserves part of each fitting partition to monitor accuracy, rather than training loss. It stops when that accuracy fails to improve for `n_iter_no_change` epochs and restores the best weights. The preprocessor has already seen this internal subset, unlike the external validation fold. The sweep keeps the default internal fraction, 0.1. The candidate list is `[False, True]`.

__Step 44:__ Set the `early_stopping` values.

In [ ]:
# The values of this one setting; every other setting stays at the reference.
mlp_early_stopping_values = [False, True]
# One row per value will hold its scores, its seconds and its fitted model.
mlp_early_stopping_rows = []

__Step 45:__ Score each candidate and keep its fitted model.

In [ ]:
for value in tqdm(mlp_early_stopping_values, desc='early_stopping values', leave=True, position=0):
    started = time.perf_counter()
    settings = {**fixed_settings, 'early_stopping': value}
    train_scores, validation_scores, fold_warnings = ([], [], [])
    for fold in mlp_prepared:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter('always', ConvergenceWarning)
            fitted = MLPClassifier(**settings).fit(fold['X_fit'], fold['y_fit'])
        note_fit(fitted, caught, f'early_stopping={value}, fold')
        train_scores.append(f1_score(fold['y_fit'], fitted.predict(fold['X_fit'])))
        validation_scores.append(f1_score(fold['y_score'], fitted.predict(fold['X_score'])))
        fold_warnings.append(any((issubclass(w.category, ConvergenceWarning) for w in caught)))
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always', ConvergenceWarning)
        deployed = MLPClassifier(**settings).fit(Z_train, y_train)
    note_fit(deployed, caught, f'early_stopping={value}, all training rows')
    shown = {key: item for key, item in MLPClassifier(**settings).get_params().items() if item != MLPClassifier().get_params()[key] or key in ('activation', 'hidden_layer_sizes', 'max_iter')}
    mlp_early_stopping_rows.append({'configuration': 'MLPClassifier(' + ', '.join((f'{key}={item!r}' for key, item in sorted(shown.items()))) + ')', 'setting': str(value), 'seconds': time.perf_counter() - started, 'train_F1': np.mean(train_scores), 'cv_F1': np.mean(validation_scores), 'test_F1': f1_score(y_test, deployed.predict(Z_test)), 'iterations': deployed.n_iter_, 'loss': deployed.loss_, 'fold warnings': sum(fold_warnings), 'train': train_scores, 'validation': validation_scores, 'parameters': settings, 'model': deployed})

__Step 46:__ Assemble and inspect the candidate scores.

In [ ]:
# The sweep table: one row per value, with the mean fold scores, the test score, the fit diagnostics and the seconds spent.
mlp_early_stopping_sweep = pd.DataFrame(mlp_early_stopping_rows)
display(mlp_early_stopping_sweep.set_index("setting")[["train_F1", "cv_F1", "test_F1", "iterations", "loss", "fold warnings", "seconds"]].round(4))
display(pd.DataFrame(fit_diagnostics).tail(11))

In [ ]:
# The sweep's winner joins the log.
mlp_early_stopping_winner = mlp_early_stopping_sweep.loc[mlp_early_stopping_sweep["cv_F1"].idxmax()]
record(mlp_log, {"frame": type(mlp_folds).__name__, "rows": len(y_train), "folds": mlp_folds.get_n_splits(),
    "name": mlp_early_stopping_winner["configuration"], "train": mlp_early_stopping_winner["train"],
    "validation": mlp_early_stopping_winner["validation"], "model": mlp_early_stopping_winner["model"]},
    Z_test, y_test, "one setting changed", metric=f1_score, label="F1")

__Step 47:__ Plot the training and validation F1 for `early_stopping`.

In [ ]:
# Training and validation F1 for each early_stopping value.
parameter_positions = np.arange(len(mlp_early_stopping_sweep))
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(parameter_positions, mlp_early_stopping_sweep["train_F1"], marker="o", color=PLOT_ORANGE, label="training")
ax.plot(parameter_positions, mlp_early_stopping_sweep["cv_F1"], marker="o", color=PLOT_BLUE, label="validation")
ax.set(title="early_stopping", xticks=parameter_positions, xticklabels=mlp_early_stopping_sweep["setting"], ylabel="F1")
ax.tick_params(axis="x", labelrotation=20)
ax.legend()
plt.tight_layout()
plt.show()

**Early stopping shortens the full fit from 200 iterations to 71, while validation F1 falls from 0.8523 to 0.8204.** Its test F1 is 0.8485, against 0.8509 without it.

__Step 48:__ Set the checkpoint budgets and candidate settings.

In [ ]:
curve_fold = mlp_prepared[0]
curve_budgets = [1, 3, 10, 30, 100, 300, 1000, 2000]
curve_points = {}
curve_values = [False, True]
curve_base = clone(MLPClassifier(**fixed_settings))

__Step 49:__ Fit each checkpoint once and measure both losses and task scores.

In [ ]:
for curve_value in tqdm(curve_values, desc='checkpoint values', leave=True, position=0):
    curve_rows = []
    for curve_k in curve_budgets:
        curve_settings = {"early_stopping": curve_value, "max_iter": curve_k}
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", ConvergenceWarning)
            curve_fit = clone(curve_base).set_params(**curve_settings).fit(
                curve_fold["X_fit"], curve_fold["y_fit"])
        note_fit(curve_fit, caught, f"early_stopping: {curve_value}, budget {curve_k}")
        curve_row = prediction_diagnostics(curve_fit, curve_fold)
        curve_rows.append([curve_k, *curve_row])
        curve_network = curve_fit
        if curve_network.n_iter_ < curve_k:
            break
    curve_points[str(curve_value)] = np.array(curve_rows)

__Step 50:__ Compare prediction loss and task score in separate panels.

In [ ]:
curve_fig, curve_axes = plt.subplots(1, 2, figsize=(12, 4.5))
for curve_label, curve_rows in curve_points.items():
    curve_line, = curve_axes[0].plot(curve_rows[:, 0], curve_rows[:, 1], label=f"{curve_label} train")
    curve_colour = curve_line.get_color()
    curve_axes[0].plot(curve_rows[:, 0], curve_rows[:, 2], "--", color=curve_colour, label=f"{curve_label} validation")
    curve_axes[1].plot(curve_rows[:, 0], curve_rows[:, 3], color=curve_colour, label=f"{curve_label} train")
    curve_axes[1].plot(curve_rows[:, 0], curve_rows[:, 4], "--", color=curve_colour, label=f"{curve_label} validation")
for curve_ax in curve_axes:
    curve_ax.set_xscale("log")
    curve_ax.set_xlabel("iteration allowance (log scale)")
    curve_ax.legend(fontsize=8)
curve_axes[0].set(ylabel="binary cross-entropy", title="Prediction loss: training and validation")
curve_axes[1].set(ylabel="F1", title="Task score: training and validation")
curve_fig.suptitle('early_stopping')
plt.tight_layout()
plt.show()
display(pd.DataFrame(fit_diagnostics).tail(sum(len(rows) for rows in curve_points.values())))

Early stopping watches accuracy on its internal subset, while the right panel shows F1 on the external fold. The left panel scores the restored weights with the same loss on the fold's training and validation rows.

### `MLP: validation_fraction`

`validation_fraction` sets the size of the internal subset. A larger fraction gives the stopping check more rows while leaving fewer rows for weight updates. The setting matters only when early stopping is enabled; it has no role for L-BFGS. Both fraction rows enable early stopping. The candidate list is `[0.1, 0.2]`.

__Step 51:__ Set the `validation_fraction` values.

In [ ]:
# The values of this one setting; every other setting stays at the reference.
mlp_validation_fraction_values = [0.1, 0.2]
# One row per value will hold its scores, its seconds and its fitted model.
mlp_validation_fraction_rows = []

__Step 52:__ Score each candidate and keep its fitted model.

In [ ]:
for value in tqdm(mlp_validation_fraction_values, desc='validation_fraction values', leave=True, position=0):
    started = time.perf_counter()
    settings = {**fixed_settings, 'early_stopping': True, 'validation_fraction': value}
    train_scores, validation_scores, fold_warnings = ([], [], [])
    for fold in mlp_prepared:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter('always', ConvergenceWarning)
            fitted = MLPClassifier(**settings).fit(fold['X_fit'], fold['y_fit'])
        note_fit(fitted, caught, f'validation_fraction={value}, fold')
        train_scores.append(f1_score(fold['y_fit'], fitted.predict(fold['X_fit'])))
        validation_scores.append(f1_score(fold['y_score'], fitted.predict(fold['X_score'])))
        fold_warnings.append(any((issubclass(w.category, ConvergenceWarning) for w in caught)))
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always', ConvergenceWarning)
        deployed = MLPClassifier(**settings).fit(Z_train, y_train)
    note_fit(deployed, caught, f'validation_fraction={value}, all training rows')
    shown = {key: item for key, item in MLPClassifier(**settings).get_params().items() if item != MLPClassifier().get_params()[key] or key in ('activation', 'hidden_layer_sizes', 'max_iter')}
    mlp_validation_fraction_rows.append({'configuration': 'MLPClassifier(' + ', '.join((f'{key}={item!r}' for key, item in sorted(shown.items()))) + ')', 'setting': str(value), 'seconds': time.perf_counter() - started, 'train_F1': np.mean(train_scores), 'cv_F1': np.mean(validation_scores), 'test_F1': f1_score(y_test, deployed.predict(Z_test)), 'iterations': deployed.n_iter_, 'loss': deployed.loss_, 'fold warnings': sum(fold_warnings), 'train': train_scores, 'validation': validation_scores, 'parameters': settings, 'model': deployed})

__Step 53:__ Assemble and inspect the candidate scores.

In [ ]:
# The sweep table: one row per value, with the mean fold scores, the test score, the fit diagnostics and the seconds spent.
mlp_validation_fraction_sweep = pd.DataFrame(mlp_validation_fraction_rows)
display(mlp_validation_fraction_sweep.set_index("setting")[["train_F1", "cv_F1", "test_F1", "iterations", "loss", "fold warnings", "seconds"]].round(4))
display(pd.DataFrame(fit_diagnostics).tail(11))

In [ ]:
# The sweep's winner joins the log.
mlp_validation_fraction_winner = mlp_validation_fraction_sweep.loc[mlp_validation_fraction_sweep["cv_F1"].idxmax()]
record(mlp_log, {"frame": type(mlp_folds).__name__, "rows": len(y_train), "folds": mlp_folds.get_n_splits(),
    "name": mlp_validation_fraction_winner["configuration"], "train": mlp_validation_fraction_winner["train"],
    "validation": mlp_validation_fraction_winner["validation"], "model": mlp_validation_fraction_winner["model"]},
    Z_test, y_test, "one setting changed", metric=f1_score, label="F1")

__Step 54:__ Plot the training and validation F1 for `validation_fraction`.

In [ ]:
# Training and validation F1 for each validation_fraction value.
parameter_positions = np.arange(len(mlp_validation_fraction_sweep))
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(parameter_positions, mlp_validation_fraction_sweep["train_F1"], marker="o", color=PLOT_ORANGE, label="training")
ax.plot(parameter_positions, mlp_validation_fraction_sweep["cv_F1"], marker="o", color=PLOT_BLUE, label="validation")
ax.set(title="validation_fraction", xticks=parameter_positions, xticklabels=mlp_validation_fraction_sweep["setting"], ylabel="F1")
ax.tick_params(axis="x", labelrotation=20)
ax.legend()
plt.tight_layout()
plt.show()

**With early stopping on, a fraction of 0.2 gives validation F1 0.8286, above 0.8204 at 0.1.** Test F1 also rises, from 0.8485 to 0.8516, while the full fit takes 93 iterations against 71.

# <font color='#E8800A'>Nested cross-validation</font> <a class="anchor" id="nested"></a>
[Back to TOC](#toc)

Nested cross-validation scores the joint search on the ten outer folds before the final search is fitted.

The domain combines the activation functions, iteration allowances and architectures of Week 12 with the optimisation settings compared here, more combinations than a class can fit. The search therefore draws sixty of them uniformly, without replacement.

This is how `RandomizedSearchCV` samples a finite list. Drawing the list once gives every outer fold and the final search the same sixty candidates, and lets `CachedSearchCV` reuse fold preprocessing. Bergstra and Bengio (2012) explain why random search can explore a useful range of values with a fixed trial budget.

__Step 55:__ Define the conditional domain and draw 60 of its combinations without replacement.

In [ ]:
mlp_activation_values = ["identity", "logistic", "tanh", "relu"]
mlp_hidden_layer_sizes_values = [(2,), (8,), (16,), (64,), (100,), (16, 16), (64, 32), (16, 16, 16), (64, 32, 16), (64, 64, 64), (32, 64, 32), (32, 64)]
mlp_max_iter_values = [20, 200, 1000, 2000]
mlp_batch_size_values = [64, 200, 512]

In [ ]:
# The joint search combines the Week 12 choices with these optimisation settings.
common_domain = {
    "estimator__activation": mlp_activation_values,
    "estimator__hidden_layer_sizes": mlp_hidden_layer_sizes_values,
    "estimator__max_iter": mlp_max_iter_values,
    "estimator__alpha": mlp_alpha_values,
}
stochastic_domain = {
    "estimator__learning_rate_init": mlp_learning_rate_init_values,
    "estimator__batch_size": mlp_batch_size_values,
    "estimator__n_iter_no_change": mlp_n_iter_no_change_values,
}
# L-BFGS has no stochastic settings; Adam and SGD each branch on early stopping, and the schedule applies to SGD only.
conditional_domain = [{**common_domain, "estimator__solver": [solver for solver in mlp_solver_values if solver == "lbfgs"]}]
for solver in [solver for solver in mlp_solver_values if solver != "lbfgs"]:
    solver_domain = {**common_domain, **stochastic_domain, "estimator__solver": [solver]}
    if solver == "sgd":
        solver_domain["estimator__learning_rate"] = mlp_learning_rate_values
    conditional_domain.append({**solver_domain, "estimator__early_stopping": [value for value in mlp_early_stopping_values if not value]})
    conditional_domain.append({**solver_domain, "estimator__early_stopping": [value for value in mlp_early_stopping_values if value],
                           "estimator__validation_fraction": mlp_validation_fraction_values})
mlp_param_grid = conditional_domain
full_domain = ParameterGrid(mlp_param_grid)
candidate_positions = (np.random.RandomState(33).choice(len(full_domain), size=60, replace=False)
                       if len(full_domain) > 60 else range(len(full_domain)))

__Step 56:__ Assemble the sampled candidate configurations.

In [ ]:
mlp_candidates = [{name: [value] for name, value in full_domain[int(position)].items()}
                  for position in candidate_positions]

__Step 57:__ Count the full domain and the sample by solver.

In [ ]:
print(f"Full valid domain: {len(full_domain):,} configurations")
print(f"Fixed sample: {len(mlp_candidates)} unique configurations")
domain_by_solver = {solver: sum(len(ParameterGrid(branch)) for branch in mlp_param_grid
                                if branch["estimator__solver"] == [solver])
                    for solver in ["lbfgs", "adam", "sgd"]}
sample_by_solver = pd.Series([candidate["estimator__solver"][0] for candidate in mlp_candidates]).value_counts()
display(pd.DataFrame({"full domain": domain_by_solver, "sample": sample_by_solver}).fillna(0).astype(int))

The conditional branches contain only applicable settings. Learning-rate schedules occur only with SGD; stochastic controls occur only with Adam or SGD; internal fractions occur only with early stopping. Every valid configuration has the same sampling probability, so branches with more combinations receive more probability mass. A small branch can receive no sampled candidate.

**The conditional domain holds 187,200 configurations: 576 for L-BFGS, 46,656 for Adam and 139,968 for SGD.** The fixed sample contains 15 Adam and 45 SGD configurations, with none from L-BFGS. Its sixty candidates cover 0.0321% of the domain, and 35 enable early stopping.

__Step 58:__ Count the early-stopping candidates and the share of the domain the sample covers.

In [ ]:
early_count = sum(candidate.get("estimator__early_stopping", [False])[0] for candidate in mlp_candidates)
print(f"early stopping candidates: {early_count}")
print(f"domain evaluated: {len(mlp_candidates) / len(full_domain):.4%}")

Each outer training part runs the sixty-candidate search on three shuffled stratified inner folds. The Week 12 joint reference uses the same ten outer folds, so the two are compared fold by fold.

The search silences the scikit-learn iteration-limit warning; the sweeps above show where it occurs. A high validation score can come from unfinished optimisation.

In [ ]:
inner_folds = StratifiedKFold(n_splits=3, shuffle=True, random_state=random_state)
base_mlp = PreparedEstimator(FoldPreprocessor(classification_recipe), MLPClassifier(**fixed_settings))
mlp_template = CachedSearchCV(GridSearchCV(base_mlp, mlp_candidates, scoring="f1", cv=inner_folds, n_jobs=2, error_score="raise"))

__Step 59:__ Score the search on the outer folds.

In [ ]:
with warnings.catch_warnings():
    search_warning_policy()
    with parallel_config(backend="loky", initializer=search_warning_policy):
        mlp_nested_fold_results = [cross_validate(mlp_template, X_train, y_train, cv=[_outer_split], scoring='f1', return_train_score=True, return_estimator=True, n_jobs=1)
            for _outer_split in tqdm(list(mlp_folds.split(X_train, y_train)), total=mlp_folds.get_n_splits(X_train, y_train), desc='Nested CV: outer folds', unit='fold', leave=True, position=0)]
        mlp_nested = {key: np.concatenate([np.asarray(result[key], dtype=object if key == 'estimator' else None) for result in mlp_nested_fold_results]) for key in mlp_nested_fold_results[0]}
        paired_gain = mlp_nested["test_score"] - np.asarray(reference["validation"])
        print(f"Week 12 joint reference CV F1: {np.mean(reference['validation']):.4f}")
        print(f"nested search CV F1: {mlp_nested['test_score'].mean():.4f}")
        print(f"outer-fold SD: {mlp_nested['test_score'].std(ddof=1):.4f}")
        print(f"paired mean gain: {paired_gain.mean():+.4f}")
        print(f"search higher in {(paired_gain > 0).sum()} of {len(paired_gain)} folds")
    nested_auc, nested_balanced = [], []
    for fitted_search, (_, score_at) in zip(mlp_nested["estimator"], mlp_folds.split(X_train, y_train)):
        heldout = X_train.iloc[score_at]
        truth = y_train.iloc[score_at]
        nested_auc.append(roc_auc_score(truth, fitted_search.predict_proba(heldout)[:, 1]))
        nested_balanced.append(balanced_accuracy_score(truth, fitted_search.predict(heldout)))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for reference_score, search_score in zip(reference["validation"], mlp_nested["test_score"]):
    ax.plot([0, 1], [reference_score, search_score], color="0.7", linewidth=1)
ax.scatter(np.zeros(10), reference["validation"], label="Week 12 joint reference", zorder=3)
ax.scatter(np.ones(10), mlp_nested["test_score"], label="Nested search", zorder=3)
ax.set(xticks=[0, 1], xticklabels=["Week 12 joint reference", "Nested search"], ylabel="Outer validation F1")
ax.legend()
plt.tight_layout()
plt.show()

**Nested search reaches mean F1 0.8515, against 0.8523 for the Week 12 joint reference.** The paired mean change is -0.0008, with search higher on five of ten folds and an outer-score standard deviation of 0.0162. The reference was itself selected on these folds, so its 0.8523 is optimistic.

# <font color='#E8800A'>The final search and its winner</font> <a class="anchor" id="search"></a>
[Back to TOC](#toc)

The final search runs the same sixty candidates and inner-fold rule on every training athlete and selects the configuration with the highest mean inner F1.

In [ ]:
with warnings.catch_warnings():
    search_warning_policy()
    with parallel_config(backend='loky', initializer=search_warning_policy):
        mlp_search = clone(mlp_template).fit(X_train, y_train)

__Step 60:__ Read the five best candidates in inner-CV order, with their held-out F1.

In [ ]:
with warnings.catch_warnings():
    search_warning_policy()
    ranking = pd.DataFrame({'settings': mlp_search.cv_results_['params'], 'inner F1': mlp_search.cv_results_['mean_test_score']})
    top_five = ranking.nlargest(5, 'inner F1').copy()
    top_five['test F1'] = [f1_score(y_test, clone(mlp_search.search_.estimator).set_params(**settings).fit(X_train, y_train).predict(X_test)) for settings in tqdm(top_five['settings'], desc='top five candidates', leave=True, position=0)]
    display(top_five)
    print(f'best inner F1: {mlp_search.best_score_:.4f}')
    print('best parameters:', mlp_search.best_params_)

__Step 61:__ Read the winning constructor and its fitted model.

In [ ]:
winner_settings = {**fixed_settings,
    **{name.removeprefix("estimator__"): value for name, value in mlp_search.best_params_.items()}}
winner_constructor = MLPClassifier(**winner_settings)
winner_model = mlp_search.best_estimator_
print(winner_constructor)
print(f"iterations completed: {winner_model.estimator_.n_iter_}")
print(f"held-out winner F1: {f1_score(y_test, winner_model.predict(X_test)):.4f}")

__Step 62:__ Score the winner on the shared folds, then log its row and the nested row.

In [ ]:
with warnings.catch_warnings(record=True) as winner_caught:
    warnings.simplefilter('always', ConvergenceWarning)
    winner_train, winner_validation = ([], [])
    winner_auc, winner_balanced = ([], [])
    for fold in tqdm(mlp_prepared, desc='Course-fold assessment', unit='fold', leave=True, position=0):
        fitted_fold = clone(winner_constructor).fit(fold['X_fit'], fold['y_fit'])
        winner_train.append(f1_score(fold['y_fit'], fitted_fold.predict(fold['X_fit'])))
        winner_validation.append(f1_score(fold['y_score'], fitted_fold.predict(fold['X_score'])))
        winner_auc.append(roc_auc_score(fold['y_score'], fitted_fold.predict_proba(fold['X_score'])[:, 1]))
        winner_balanced.append(balanced_accuracy_score(fold['y_score'], fitted_fold.predict(fold['X_score'])))
note_fit(fitted_fold, winner_caught, 'winner, shared folds')
winner_warnings = sum((issubclass(w.category, ConvergenceWarning) for w in winner_caught))
print(f'Winner convergence warnings on the shared folds: {winner_warnings}')
winner_name = ' '.join(repr(winner_constructor).split())

In [ ]:
record(mlp_log, {"frame": type(mlp_folds).__name__, "rows": len(y_train),
                 "folds": mlp_folds.get_n_splits(), "name": winner_name,
                 "train": winner_train, "validation": winner_validation, "model": winner_model},
       X_test, y_test, "winner's own score", metric=f1_score, label="F1")

In [ ]:
record(mlp_log, {"frame": type(mlp_folds).__name__, "rows": len(y_train),
                 "folds": mlp_folds.get_n_splits(), "name": f"joint search over {len(mlp_candidates)} sampled combinations",
                 "train": mlp_nested["train_score"], "validation": mlp_nested["test_score"],
                 "model": winner_model},
       X_test, y_test, "nested estimate", metric=f1_score, label="F1")

**The final search selects three ReLU layers `(32, 64, 32)` with SGD, batch size 64, `alpha=0.01` and initial learning rate 0.01.** Early stopping uses a 0.1 validation fraction and patience 5; the full fit stops after 20 of the 200 allowed iterations. Its inner F1 is 0.8534 and test F1 0.8549. Its own shared-fold F1 is 0.8473, with no convergence warning, while the nested estimate is 0.8515.

# <font color='#E8800A'>The course board</font> <a class="anchor" id="board"></a>
[Back to TOC](#toc)

The board reports nested F1 for the search procedure and test F1 for its final model. It includes the Week 12 joint network for comparison.

In [ ]:
results_dir = "../../results"
floor = f1_score(y_test, np.full(len(y_test), y_train.mode().iat[0]))
board = Scoreboard("champions outcome", "F1", floor=floor, context=result_context(
    "../../data/interim/champions.csv", classification_recipe_log,
    y_train.index.tolist(), y_test.index.tolist(), outer_folds=outer_folds(mlp_folds, X_train, y_train)))
for week, model, cv, test in results_before(13, "classification", results_dir,
                                          metric=board.metric, context=board.context):
    board.record(week, model, cv, test)
board.record(13, "MLP search (60 fixed candidates, nested)",
             mlp_nested["test_score"].mean(), f1_score(y_test, winner_model.predict(X_test)))
board.save(results_dir, 13, "classification", baseline_cv=np.mean(reference["validation"]),
           baseline_test=f1_score(y_test, reference["model"].predict(Z_test)))
print(board.table())
chosen_model = winner_model
joblib.dump(chosen_model, f"{results_dir}/week_13_classification_model.joblib")

# ROC-AUC and balanced accuracy for the same three models; F1 alone chose the winner.
companion_metrics = pd.DataFrame([
    {"model": 'Week 12 joint reference', "CV F1": np.mean(reference_validation), "test F1": f1_score(y_test, fitted_reference.predict(Z_test)), "CV ROC-AUC": np.mean(reference_auc), "test ROC-AUC": roc_auc_score(y_test, fitted_reference.predict_proba(Z_test)[:, 1]), "CV balanced accuracy": np.mean(reference_balanced), "test balanced accuracy": balanced_accuracy_score(y_test, fitted_reference.predict(Z_test))},
    {"model": 'Search winner, own shared-fold score', "CV F1": np.mean(winner_validation), "test F1": f1_score(y_test, winner_model.predict(X_test)), "CV ROC-AUC": np.mean(winner_auc), "test ROC-AUC": roc_auc_score(y_test, winner_model.predict_proba(X_test)[:, 1]), "CV balanced accuracy": np.mean(winner_balanced), "test balanced accuracy": balanced_accuracy_score(y_test, winner_model.predict(X_test))},
    {"model": 'Search, nested estimate', "CV F1": np.mean(mlp_nested["test_score"]), "test F1": f1_score(y_test, winner_model.predict(X_test)), "CV ROC-AUC": np.mean(nested_auc), "test ROC-AUC": roc_auc_score(y_test, winner_model.predict_proba(X_test)[:, 1]), "CV balanced accuracy": np.mean(nested_balanced), "test balanced accuracy": balanced_accuracy_score(y_test, winner_model.predict(X_test))},
])
display(companion_metrics)

**The search enters the board at nested F1 0.8515 and test F1 0.8549, against 0.8523 and 0.8509 for the Week 12 joint reference.** The two estimates rank these rows differently.

# <font color='#E8800A'>Key takeaways</font> <a class="anchor" id="takeaways"></a>
[Back to TOC](#toc)

1. Optimisation, regularisation, capacity and stopping settings affect different parts of fitting. Applicable combinations depend on the solver.
2. Backpropagation applies the chain rule from the loss back to every weight and bias, using the parameter values before the update.
3. One-setting sweeps describe each setting's effect; the joint search combines every tested value.
4. Random search draws sixty candidates from a domain too large to fit exhaustively.

### The Week 13 winner

The drawing uses the fitted network selected by the final training search. Layer sizes and activations come from its fitted arrays; ellipses mark units omitted from the drawing.

__Step 63:__ Read the fitted arrays and the activation function of each layer, and choose the units to draw.

In [ ]:
# The fitted network's own arrays. A layer wider than six units shows its first five units, an ellipsis and its last unit.
net_model = winner_model.estimator_
net_weights, net_biases = net_model.coefs_, net_model.intercepts_
net_sizes = [net_weights[0].shape[0]] + [weight.shape[1] for weight in net_weights]
net_shown = [list(range(size)) if size <= 6 else [*range(5), None, size - 1] for size in net_sizes]
# The activation of each layer is read from the fitted model: `activation` for the hidden layers and `out_activation_` for the output.
net_activations = ["none"] + [net_model.activation] * (len(net_sizes) - 2) + [net_model.out_activation_]
net_symbol = "p"
net_layers = pd.DataFrame({"units": net_sizes, "units drawn": [len([unit for unit in shown if unit is not None]) for shown in net_shown],
                           "activation": net_activations},
                          index=["input"] + [f"hidden {layer}" for layer in range(1, len(net_sizes) - 1)] + ["output"])
display(net_layers)

__Step 64:__ Place the drawn units and name their nodes.

In [ ]:
# One row per unit drawn, and one row per connection between units drawn in neighbouring layers.
net_gap = 4
net_prefix = ["x"] + (["h"] if len(net_sizes) == 3 else [f"h{layer}_" for layer in range(1, len(net_sizes) - 1)])
net_math = ["x"] + (["h"] if len(net_sizes) == 3 else [f"h^{{({layer})}}" for layer in range(1, len(net_sizes) - 1)])
net_node_rows = []
for layer, slots in enumerate(net_shown):
    last = layer == len(net_sizes) - 1
    for slot, unit in enumerate(slots):
        net_node_rows.append({"layer": layer, "unit": -1 if unit is None else unit, "x": layer * net_gap, "y": 1.2 * ((len(slots) - 1) / 2 - slot),
                              "name": "output" if last else f"{net_prefix[layer]}{unit + 1}" if unit is not None else "",
                              "tex": f"${net_symbol}$" if last else f"${net_math[layer]}_{{{unit + 1}}}$" if unit is not None else ""})
net_nodes = pd.DataFrame(net_node_rows)
net_circles = net_nodes[net_nodes["unit"] >= 0]

__Step 65:__ Extract the fitted weights connecting the drawn units.

In [ ]:
net_link_rows = []
for layer, weights in enumerate(net_weights):
    for source in net_circles[net_circles["layer"] == layer].itertuples():
        for target in net_circles[net_circles["layer"] == layer + 1].itertuples():
            net_link_rows.append({"from": source.name, "to": target.name, "weight": weights[source.unit, target.unit],
                                  "x0": source.x, "y0": source.y, "x1": target.x, "y1": target.y})
net_links = pd.DataFrame(net_link_rows)
net_largest = max(np.abs(weight).max() for weight in net_weights)

__Step 66:__ Label each layer with its fitted activation and array shapes.

In [ ]:
# The text of the figure: a title for each layer with its activation function, and the shapes of each pair of arrays.
net_hidden_formula = {"relu": r"\max(0,z)", "tanh": r"\tanh(z)", "logistic": r"\sigma(z)", "identity": "z"}[net_model.activation]
net_output_formula = {"logistic": r"\sigma(a)", "identity": "a"}[net_model.out_activation_]
net_hidden_names = ["hidden layer" if len(net_sizes) == 3 else f"hidden layer {layer}" for layer in range(1, len(net_sizes) - 1)]
net_titles = ([f"inputs $x$\n{net_sizes[0]} units"]
              + [f"{name}, {size} units\n{net_model.activation}: $h={net_hidden_formula}$" for name, size in zip(net_hidden_names, net_sizes[1:-1])]
              + [f"output\n{net_model.out_activation_}: ${net_symbol}={net_output_formula}$" + ""])
net_captions = [f"$W_{{{k}}}$ ({w.shape[0]}×{w.shape[1]}), $b_{{{k}}}$ ({len(b)})" for k, (w, b) in enumerate(zip(net_weights, net_biases), 1)]
net_face = ["#EDEDED"] + ["#FFF3C4"] * (len(net_sizes) - 2) + ["#DFF5E8"]
print(f"{len(net_links)} connections drawn; largest absolute weight of the network {net_largest:.3f}")
# The ten largest of the connections drawn.
display(net_links.assign(size=net_links["weight"].abs()).nlargest(10, "size")[["from", "to", "weight"]].round(3).reset_index(drop=True))

__Step 67:__ Draw the network from that table.

In [ ]:
net_blue, net_orange = "#1779A8", "#C96A00"
net_fig, net_ax = plt.subplots(figsize=(min(14, 2.6 * len(net_sizes) + 2), 7.4))
for link in net_links.itertuples():
    net_size = np.abs(link.weight) / net_largest
    net_ax.plot([link.x0, link.x1], [link.y0, link.y1], color=net_blue if link.weight > 0 else net_orange,
                lw=0.4 + 4.6 * net_size, alpha=0.12 + 0.8 * net_size, solid_capstyle="round", zorder=1)
net_ax.scatter(net_circles["x"], net_circles["y"], s=560, facecolor=[net_face[layer] for layer in net_circles["layer"]],
               edgecolor="#222222", linewidth=1.2, zorder=3)
for node in net_nodes.itertuples():
    net_ax.text(node.x, node.y, node.tex if node.unit >= 0 else r"$\vdots$", ha="center", va="center",
                fontsize=11 if node.unit >= 0 else 18, zorder=4)
net_top, net_bottom = net_nodes["y"].max(), net_nodes["y"].min()
for layer, title in enumerate(net_titles):
    net_ax.text(layer * net_gap, net_top + 0.9, title, ha="center", va="bottom", fontweight="bold", fontsize=12)
for gap, caption in enumerate(net_captions):
    net_ax.text((gap + 0.5) * net_gap, net_bottom - 1.0, caption, ha="center", va="top", fontsize=12)
# Two keys: the sign, and three sample widths.
net_sign_key = net_ax.legend(handles=[net_ax.plot([], [], color=net_blue, lw=3, label="positive")[0], net_ax.plot([], [], color=net_orange, lw=3, label="negative")[0]],
                             title="weight sign", loc="lower left", bbox_to_anchor=(0.0, -0.06), ncol=2, frameon=False, fontsize=11, title_fontsize=11)
net_ax.add_artist(net_sign_key)
net_ax.legend(handles=[net_ax.plot([], [], color="#555555", lw=0.4 + 4.6 * size, label=f"{size * net_largest:.2f}")[0] for size in (0.25, 0.5, 1.0)],
              title="weight size", loc="lower right", bbox_to_anchor=(1.0, -0.06), ncol=3, frameon=False, fontsize=11, title_fontsize=11)
net_ax.set(xlim=(-1, net_gap * (len(net_sizes) - 1) + 1.4), ylim=(net_bottom - 2.8, net_top + 3.4), title="The final search winner")
net_ax.axis("off")
plt.tight_layout()

__Step 68:__ Show the figure.

In [ ]:
plt.show()

**The final winner has hidden layers of 32, 64 and 32 ReLU units and a logistic output.** Its weight matrices are 28 by 32, 32 by 64, 64 by 32 and 32 by 1. The largest absolute weight is 0.698.

# <font color='#E8800A'>References</font> <a class="anchor" id="references"></a>
[Back to TOC](#toc)

- Bergstra, J. and Bengio, Y. (2012). [Random Search for Hyper-Parameter Optimization](https://www.jmlr.org/papers/v13/bergstra12a.html). Journal of Machine Learning Research, 13, 281-305.
- Robbins, H. and Monro, S. (1951). [A Stochastic Approximation Method](https://doi.org/10.1214/aoms/1177729586). The Annals of Mathematical Statistics, 22(3), 400-407.
- Kingma, D. P. and Ba, J. (2015). [Adam: A Method for Stochastic Optimization](https://arxiv.org/abs/1412.6980). ICLR.
- Liu, D. C. and Nocedal, J. (1989). [On the limited memory BFGS method for large scale optimization](https://doi.org/10.1007/BF01589116). Mathematical Programming, 45, 503-528.
- scikit-learn, [MLPClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html).
- scikit-learn, [RandomizedSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html). Finite lists are sampled without replacement.
- scikit-learn, [Nested versus non-nested cross-validation](https://scikit-learn.org/stable/auto_examples/model_selection/plot_nested_cross_validation_iris.html).